### ЗАДАЧА: Триаж обращений службы поддержки

Команда поддержки получает пакет строк с обращениями от разных сервисов.
Нужно обработать их так, чтобы:
- корректные обращения попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, в каких каналах остались не подтверждённые обращения,
- а также какова средняя длительность обработки по уровням приоритета.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестный уровень приоритета или канал,
а часть передаёт неправильный флаг подтверждения.
        


In [1]:
import math

rows = [
    'INC-100|checkout|critical|12|pager|yes',
    'INC-101|search|high|7|slack|no',
    'INC-102|billing|medium|zero|email|yes',
    'INC-103|video|critical|-3|pager|no',
    'INC-104|feed|warning|5|slack|yes',
    'INC-105|auth|low|2|slack|no',
    'INC-106|cdn|high|4|email|maybe',
    'INC-107|ml|medium|9|slack|no',
]


class IncidentProcessingError(Exception):
    pass


class IncidentFormatError(IncidentProcessingError):
    pass


class SeverityError(IncidentProcessingError):
    pass


class DurationError(IncidentProcessingError):
    pass


class ChannelError(IncidentProcessingError):
    pass


class AcknowledgedFlagError(IncidentProcessingError):
    pass


def parse_incident(row):
    parts = [p.strip() for p in row.split('|')]
    if len(parts) != 6:
        raise IncidentFormatError(f"Expected 6 parts, but got {len(parts)} in row: {row}")

    incident_id, service, severity, duration_raw, channel, acknowledged_raw = parts

    try:
        duration = float(duration_raw)
        if duration <= 0:
            raise DurationError(f"Duration must be positive, but got {duration_raw} in row: {row}")
    except ValueError as exc:
        raise DurationError(f"Invalid duration format: {duration_raw} in row: {row}") from exc

    allowed_severities = {'low', 'medium', 'high', 'critical'}
    if severity not in allowed_severities:
        raise SeverityError(f"Invalid severity: {severity}. Allowed: {allowed_severities} in row: {row}")

    allowed_channels = {'email', 'slack', 'pager'}
    if channel not in allowed_channels:
        raise ChannelError(f"Invalid channel: {channel}. Allowed: {allowed_channels} in row: {row}")

    if acknowledged_raw not in {'yes', 'no'}:
        raise AcknowledgedFlagError(f"Invalid acknowledged flag: {acknowledged_raw}. Allowed: {{'yes', 'no'}} in row: {row}")
    acknowledged = acknowledged_raw == 'yes'

    return {
        'incident_id': incident_id,
        'service': service,
        'severity': severity,
        'duration_min': duration,  
        'channel': channel,
        'acknowledged': acknowledged,
    }


def process_batch(rows):
    incidents = []
    errors = []
    for row in rows:
        try:
            parsed = parse_incident(row)
            incidents.append(parsed)
        except IncidentProcessingError as e:
            errors.append((row, type(e).__name__, str(e)))
    return incidents, errors


def error_counts(errors):
    counts = {}
    for _, error_type, _ in errors:
        counts[error_type] = counts.get(error_type, 0) + 1
    return counts


def unacked_by_channel(incidents):
    unacked = {}
    for incident in incidents:
        if not incident['acknowledged']:
            channel = incident['channel']
            incident_id = incident['incident_id']
            if channel not in unacked:
                unacked[channel] = []
            unacked[channel].append(incident_id)
    return unacked


def average_duration_by_severity(incidents):
    severity_durations = {}
    for incident in incidents:
        severity = incident['severity']
        duration = incident['duration_min']
        if severity not in severity_durations:
            severity_durations[severity] = {'total_duration': 0, 'count': 0}
        severity_durations[severity]['total_duration'] += duration
        severity_durations[severity]['count'] += 1

    avg_durations = {}
    for severity, data in severity_durations.items():
        avg_durations[severity] = data['total_duration'] / data['count']
    return avg_durations


def longest_incident(incidents):
    if not incidents:
        return None
    return max(incidents, key=lambda x: x['duration_min'])


# TODO: вызвать process_batch(rows)
incidents, errors = process_batch(rows)

print("--- Processing Results ---")
print(f"Number of valid incidents: {len(incidents)}")
print(f"Number of errors: {len(errors)}")

print("\n--- Errors ---")
if not errors:
    print("No errors found.")
else:
    for row, error_type, message in errors:
        print(f"  Row: '{row}' | Error Type: {error_type} | Message: {message}")

# TODO: собрать error_counts: dict[str, int]
error_summary = error_counts(errors)
print("\n--- Error Summary by Type ---")
if not error_summary:
    print("No errors to summarize.")
else:
    for error_type, count in error_summary.items():
        print(f"  {error_type}: {count}")

# TODO: собрать unacked_by_channel: dict[str, list[str]] только для acknowledged == False
unacked_incidents = unacked_by_channel(incidents)
print("\n--- Unacknowledged Incidents by Channel ---")
if not unacked_incidents:
    print("No unacknowledged incidents.")
else:
    for channel, incident_ids in unacked_incidents.items():
        print(f"  {channel}: {', '.join(incident_ids)}")

# TODO: собрать average_duration_by_severity только по вапидным строкам
avg_duration_summary = average_duration_by_severity(incidents)
print("\n--- Average Duration by Severity (in minutes) ---")
if not avg_duration_summary:
    print("No data to calculate average duration.")
else:
    for severity, avg_duration in avg_duration_summary.items():
        print(f"  {severity}: {avg_duration:.2f}")

# TODO: найти longest_incident среди валидных инцидентов по duration_min
longest = longest_incident(incidents)
print("\n--- Longest Incident ---")
if longest:
    print(f"  Incident ID: {longest['incident_id']}")
    print(f"  Service: {longest['service']}")
    print(f"  Severity: {longest['severity']}")
    print(f"  Duration: {longest['duration_min']:.2f} minutes")
else:
    print("No incidents to determine the longest.")

--- Processing Results ---
Number of valid incidents: 4
Number of errors: 4

--- Errors ---
  Row: 'INC-102|billing|medium|zero|email|yes' | Error Type: DurationError | Message: Invalid duration format: zero in row: INC-102|billing|medium|zero|email|yes
  Row: 'INC-103|video|critical|-3|pager|no' | Error Type: DurationError | Message: Duration must be positive, but got -3 in row: INC-103|video|critical|-3|pager|no
  Row: 'INC-104|feed|warning|5|slack|yes' | Error Type: SeverityError | Message: Invalid severity: warning. Allowed: {'critical', 'medium', 'low', 'high'} in row: INC-104|feed|warning|5|slack|yes
  Row: 'INC-106|cdn|high|4|email|maybe' | Error Type: AcknowledgedFlagError | Message: Invalid acknowledged flag: maybe. Allowed: {'yes', 'no'} in row: INC-106|cdn|high|4|email|maybe

--- Error Summary by Type ---
  DurationError: 2
  SeverityError: 1
  AcknowledgedFlagError: 1

--- Unacknowledged Incidents by Channel ---
  slack: INC-101, INC-105, INC-107

--- Average Duration by Se